# CMC Restaurant AI/ML & Data Mining

Notebook này phục vụ học phần **Học máy và Khai phá dữ liệu** cho dự án CMC Restaurant QR AI Ordering.

## Ranh giới quan trọng

- Notebook này **không gọi 9router**.
- Notebook này **không gọi external LLM API**.
- Chatbot sản phẩm có thể dùng LLM API và RAG ở phần app, nhưng nhóm **không huấn luyện LLM**.
- Đóng góp của notebook là dataset, preprocessing, EDA, association rule mining, content-based recommendation, baseline và evaluation.


## 1. Chuẩn bị môi trường

Trong Colab, hãy upload hoặc clone cả thư mục `coursework/ai-ml-data-mining/`, sau đó đặt working directory tại thư mục này. Notebook chỉ dùng thư viện phổ biến: `pandas`, `numpy`, `scikit-learn`, `matplotlib`.

In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path('.')
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
CHART_DIR = OUTPUT_DIR / 'charts'
OUTPUT_DIR.mkdir(exist_ok=True)
CHART_DIR.mkdir(exist_ok=True)


## 2. Đọc dataset và kiểm tra schema

Dataset gồm menu, giao dịch đơn hàng và FAQ/intent mẫu. Trường `is_available` giúp ngăn hệ thống gợi ý món tạm hết.

In [ ]:
menu = pd.read_csv(DATA_DIR / 'menu_items.csv')
orders = pd.read_csv(DATA_DIR / 'order_transactions.csv')
faq = pd.read_csv(DATA_DIR / 'faq_entries.csv')

required_menu_cols = {'menu_item_id', 'name', 'category', 'price_vnd', 'tags', 'description', 'is_available'}
required_order_cols = {'order_id', 'session_id', 'order_type', 'customer_segment', 'party_size', 'menu_item_id', 'quantity', 'unit_price_vnd'}
assert required_menu_cols.issubset(menu.columns)
assert required_order_cols.issubset(orders.columns)

menu['is_available'] = menu['is_available'].astype(bool)
orders['line_total'] = orders['quantity'] * orders['unit_price_vnd']

print('menu shape:', menu.shape)
print('orders shape:', orders.shape)
print('faq shape:', faq.shape)
menu.head()


## 3. EDA: món phổ biến, danh mục, kích thước đơn và giá trị đơn

Các thống kê này giúp giải thích dữ liệu trước khi đưa vào gợi ý.

In [ ]:
top_items = (
    orders.groupby('item_name', as_index=False)['quantity']
    .sum()
    .sort_values('quantity', ascending=False)
)

category_distribution = menu.groupby('category', as_index=False)['menu_item_id'].count()
category_distribution = category_distribution.rename(columns={'menu_item_id': 'item_count'})

order_size = orders.groupby('order_id')['menu_item_id'].nunique().rename('unique_items')
order_value = orders.groupby('order_id')['line_total'].sum().rename('order_value_vnd')
order_summary = pd.concat([order_size, order_value], axis=1).reset_index()

print('Average order value:', round(order_summary['order_value_vnd'].mean(), 0))
display(top_items.head(8))
display(category_distribution)
display(order_summary.describe())


In [ ]:
plt.figure(figsize=(8, 4))
top_items.head(6).plot(kind='barh', x='item_name', y='quantity', legend=False, color='#df6429')
plt.title('Top món theo số lượng')
plt.xlabel('Số lượng')
plt.ylabel('Món')
plt.tight_layout()
plt.savefig(CHART_DIR / 'top_items_from_notebook.png', dpi=140)

plt.figure(figsize=(8, 4))
category_distribution.plot(kind='bar', x='category', y='item_count', legend=False, color='#f39a54')
plt.title('Phân bố danh mục')
plt.xlabel('Danh mục')
plt.ylabel('Số món')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(CHART_DIR / 'category_distribution_from_notebook.png', dpi=140)


## 4. Baseline recommendation

Baseline đơn giản: gợi ý món còn bán có số lượng bán cao nhất. Baseline giúp so sánh với các cách gợi ý thông minh hơn.

In [ ]:
available_menu = menu[menu['is_available']].copy()
sales = orders.groupby('menu_item_id', as_index=False)['quantity'].sum().rename(columns={'quantity': 'sold_qty'})
popular_available = available_menu.merge(sales, on='menu_item_id', how='left').fillna({'sold_qty': 0})
popular_available = popular_available.sort_values(['sold_qty', 'price_vnd'], ascending=[False, True])
baseline_top3 = popular_available[['menu_item_id', 'name', 'category', 'sold_qty']].head(3)
display(baseline_top3)


## 5. Content-based recommendation

Đặc trưng được lấy từ category, tags, description, price range và availability. Món tạm hết luôn bị loại khỏi kết quả.

In [ ]:
def price_bucket(price):
    if price < 80000:
        return 'low_price'
    if price < 180000:
        return 'mid_price'
    return 'premium_price'

menu['price_bucket'] = menu['price_vnd'].map(price_bucket)
menu['feature_text'] = (
    menu['category'] + ' ' + menu['tags'].str.replace(';', ' ', regex=False) + ' ' +
    menu['description'] + ' ' + menu['price_bucket']
)

vectorizer = TfidfVectorizer(lowercase=True)
item_matrix = vectorizer.fit_transform(menu['feature_text'])

def recommend_content(query, k=3):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, item_matrix).ravel()
    result = menu.assign(score=scores)
    result = result[result['is_available']].sort_values('score', ascending=False)
    return result[['menu_item_id', 'name', 'category', 'price_vnd', 'score']].head(k)

display(recommend_content('hải sản tôm món nhóm share'))
display(recommend_content('đồ uống thanh mát fresh cool'))


## 6. Association rule mining

Notebook tính support, confidence và lift cho các cặp món. Cách làm thủ công để dễ giải thích trong bài báo cáo, không phụ thuộc package ngoài như `mlxtend`.

In [ ]:
transactions = orders.groupby('order_id')['item_name'].apply(lambda x: sorted(set(x))).tolist()
n_transactions = len(transactions)

def support(itemset):
    itemset = set(itemset)
    return sum(itemset.issubset(set(t)) for t in transactions) / n_transactions

rules = []
all_items = sorted(orders['item_name'].unique())
for a, b in combinations(all_items, 2):
    support_ab = support([a, b])
    if support_ab == 0:
        continue
    for antecedent, consequent in [(a, b), (b, a)]:
        support_a = support([antecedent])
        support_b = support([consequent])
        confidence = support_ab / support_a if support_a else 0
        lift = confidence / support_b if support_b else 0
        if support_ab >= 0.10 and confidence >= 0.30:
            rules.append({
                'antecedent_items': antecedent,
                'consequent_items': consequent,
                'support': round(support_ab, 3),
                'confidence': round(confidence, 3),
                'lift': round(lift, 3),
            })

rules_df = pd.DataFrame(rules).sort_values(['lift', 'confidence', 'support'], ascending=False)
rules_df.to_csv(OUTPUT_DIR / 'association_rules.csv', index=False)
display(rules_df.head(10))


## 7. Evaluation: Precision@K, Recall@K và case định tính

Với dataset nhỏ, notebook dùng các case demo có nhãn kỳ vọng để minh họa cách đánh giá.

In [ ]:
evaluation_cases = [
    {
        'case_id': 'case-001',
        'scenario': 'Khách thích hải sản',
        'query': 'hải sản tôm món nhóm share',
        'expected': {'Tôm rang muối', 'Lẩu Thái hải sản', 'Gỏi xoài tôm sú'},
        'method': 'content_based',
    },
    {
        'case_id': 'case-002',
        'scenario': 'Khách hỏi đồ uống thanh mát',
        'query': 'đồ uống thanh mát fresh cool',
        'expected': {'Trà đào cam sả'},
        'method': 'content_based',
    },
    {
        'case_id': 'case-003',
        'scenario': 'Pickup bữa trưa nhanh',
        'query': 'pickup bữa trưa phở đồ uống',
        'expected': {'Phở bò đặc biệt', 'Trà đào cam sả'},
        'method': 'baseline_popular',
    },
]

def precision_recall_at_k(recommended, expected, k=3):
    recommended_k = list(recommended)[:k]
    hit_count = len(set(recommended_k) & set(expected))
    precision = hit_count / k
    recall = hit_count / len(expected) if expected else 0
    return precision, recall

rows = []
for case in evaluation_cases:
    if case['method'] == 'baseline_popular':
        recommended = baseline_top3['name'].tolist()
    else:
        recommended = recommend_content(case['query'], k=3)['name'].tolist()
    precision, recall = precision_recall_at_k(recommended, case['expected'], k=3)
    rows.append({
        'case_id': case['case_id'],
        'scenario': case['scenario'],
        'method': case['method'],
        'recommended_items': ';'.join(recommended),
        'expected_items': ';'.join(sorted(case['expected'])),
        'precision_at_3': round(precision, 3),
        'recall_at_3': round(recall, 3),
    })

evaluation_df = pd.DataFrame(rows)
evaluation_df.to_csv(OUTPUT_DIR / 'recommendation_examples.csv', index=False)
display(evaluation_df)


## 8. Liên hệ với chatbot/RAG của app

Output của notebook có thể hỗ trợ chatbot ở issue #11 theo cách an toàn:

- Dùng `association_rules.csv` để gợi ý combo hoặc món thường đi kèm.
- Dùng `recommendation_examples.csv` để có demo case và metric khi thuyết trình.
- Dùng `menu_items.csv` làm nguồn kiểm tra món thật, giá thật, trạng thái còn món.
- Chatbot chỉ diễn giải insight; không được tự tạo món, giá hoặc tự thêm món vào giỏ nếu khách chưa xác nhận.


## 9. Kết luận và giới hạn

Notebook chứng minh phần ML/Data Mining qua dữ liệu và phương pháp có thể đánh giá được, tách biệt với phần LLM API.

Giới hạn:

- Dataset hiện là synthetic, quy mô nhỏ.
- Association rules cần nhiều đơn thật hơn để ổn định.
- Recommendation chưa cá nhân hóa theo lịch sử dài hạn.
- Notebook không triển khai production API và không gọi 9router.
